# MATH840 — Lab 8: Combining forecasts

**Week 9 | graded assignment 7 of 7 | 8 points | due 23:59 today**

The last weekly lab. Your series from Week 4 — for the fourth time. You already have an ETS model (Lab 5)
and a seasonal ARIMA (Lab 7), each with a published accuracy score on the hidden horizon. Tonight you
**rebuild them, add MSTL, and combine** — as the *Advanced Forecasting Methods* lecture and the Week 9
practice did — and then you decide whether the combination earned its place against your own past
results.

No plots of the series this time: you have read it three times. Every plot below is about the models.

**Accuracy is marked**, as in Labs 5 and 7: beat your benchmark — 3 points, within 10% — 2, worse — 1
with a written explanation.

**Keep the section headings below.** They map one-to-one onto the marking rubric:

| Section | Criteria | Points |
|---|---|---|
| 1 | Your models, rebuilt — and one new member | 1 |
| 2 | Forecast accuracy against your benchmark (scored from your file) | 3 |
| 3 | Justification block: combine or select, the interval, conclusions against your past results | 3 |
| 4 | Code quality and reproducibility | 1 |

Submit `SURNAME_lab09_forecast.csv`, `SURNAME_lab09.pdf` and `SURNAME_lab09.ipynb` to the Week 9
activity on Moodle before 23:59.

## 0. Setup

Given — run it and move on. Nothing in this section is marked.

In [ ]:
!pip install -q statsforecast==2.1.1   # the version the pool was screened with

import hashlib
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsforecast.models import ARIMA, AutoARIMA, AutoETS, MSTL

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)
print("pandas", pd.__version__)

In [ ]:
SURNAME = ""   # <-- for your submission file names
MY_ID   = ""   # <-- the SAME identifier you have used since Week 1

if not SURNAME.strip() or not MY_ID.strip():
    raise ValueError("Set SURNAME and MY_ID before running the rest.")

### Your series

The same identifier produces the same series as in Weeks 4, 6, 7 and 8. If the code printed below is not
the one from your earlier submissions, stop and check `MY_ID`.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data/trackb")


def assign_series(student_id: str, codes: list[str]) -> str:
    digest = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    return codes[int(digest, 16) % len(codes)]


index = pd.read_csv(f"{BASE}/index.csv")
CODE = assign_series(MY_ID, sorted(index["code"]))
spec = index.set_index("code").loc[CODE]

M    = int(spec["m"])       # seasonal period: 12 monthly, 4 quarterly
H    = int(spec["h"])       # how many steps you are forecasting
FREQ = spec["freq"]         # pandas frequency of the calendar: "MS" or "QS"

s = pd.read_csv(f"{BASE}/{CODE}.csv", parse_dates=["ds"])
y = s["y"].to_numpy(float)
dates = s["ds"]

print(f"series {CODE}: {len(s)} observations, {dates.min().date()} to {dates.max().date()}")
print(f"seasonal period m = {M}, forecast horizon h = {H}, frequency {FREQ}")

### Helpers

Given. The benchmark, `mase` and the validation split are the same as in every lab since Week 4.
`fit_ets` is Lab 5's, `fit_arima` is Lab 7's, `fit_mstl` is new; all three take `log=True` and then return
forecasts and intervals on the original scale.

In [ ]:
def benchmark_forecasts(history: np.ndarray, h: int, m: int) -> dict[str, np.ndarray]:
    # The four simple methods from Week 1.
    T = len(history)
    return {
        "mean":   np.repeat(history.mean(), h),
        "naive":  np.repeat(history[-1], h),
        "snaive": np.array([history[-m + (i % m)] for i in range(h)]),
        "drift":  history[-1] + np.arange(1, h + 1) * (history[-1] - history[0]) / (T - 1),
    }


SCALE = float(np.mean(np.abs(y[M:] - y[:-M])))      # MASE denominator, fixed for this series


def mase(actual, forecast) -> float:
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))) / SCALE)


def rmse(actual, forecast) -> float:
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


train, valid = y[:-H], y[-H:]
simple = benchmark_forecasts(train, H, M)
BENCHMARK = min(simple, key=lambda name: rmse(valid, simple[name]))
print(f"your benchmark: {BENCHMARK} (validation MASE {mase(valid, simple[BENCHMARK]):.3f})")

FUTURE = pd.date_range(dates.iloc[-1], periods=H + 1, freq=FREQ)[1:]
print("you are forecasting:", FUTURE[0].date(), "to", FUTURE[-1].date())

In [ ]:
class Fitted:
    # A fitted model whose forecasts are on the original scale, whatever it was fitted on.
    def __init__(self, model, log):
        self.model, self.log = model, log
        self.model_ = model.model_

    def predict(self, h: int, level=None):
        out = self.model.predict(h=h, level=level)
        return {k: np.exp(v) for k, v in out.items()} if self.log else out


def fit_ets(history, spec=None, damped=None, log=False) -> Fitted:
    # Lab 5: spec=None lets AutoETS search; "MAM", "AAdA" (damped) and so on fix the model.
    x = np.log(history) if log else np.asarray(history, float)
    if spec is None:
        return Fitted(AutoETS(season_length=M).fit(y=x), log)
    spec = str(spec).strip().upper()
    if spec[2:4] in ("DA", "DM", "DN"):
        spec, damped = spec[:2] + spec[3:], True
    return Fitted(AutoETS(season_length=M, model=spec,
                          damped=bool(damped) if spec[1] != "N" else None).fit(y=x), log)


def fit_arima(history, order=None, seasonal=(0, 0, 0), log=False, drift=False) -> Fitted:
    # Lab 7: order=(p, d, q) and seasonal=(P, D, Q) for a hand specification; order=None for AutoARIMA.
    x = np.log(history) if log else np.asarray(history, float)
    if order is None:
        model = AutoARIMA(season_length=M)
    else:
        model = ARIMA(order=tuple(order), seasonal_order=tuple(seasonal), season_length=M,
                      include_drift=drift and (order[1] + seasonal[1]) == 1)
    return Fitted(model.fit(y=x), log)


def fit_mstl(history, log=False) -> Fitted:
    # New: STL with the seasonal period M, an automatic ETS without a season for the rest.
    x = np.log(history) if log else np.asarray(history, float)
    return Fitted(MSTL(season_length=M).fit(y=x), log)


FITTERS = {"ets": fit_ets, "arima": fit_arima, "mstl": fit_mstl}


def fit_member(member, history) -> Fitted:
    kind, args = member
    return FITTERS[kind](history, **args)

In [ ]:
WINDOWS = 4     # rolling origins, h steps each; window 1 is the validation window of every earlier lab


def backtest(members: dict, history: np.ndarray, h: int = H, windows: int = WINDOWS) -> pd.DataFrame:
    # Every member from every origin: one row per (window, step), the actual value, and for each member
    # its forecast and 80% bounds. The benchmark is in the table too, by its own rule.
    frames = []
    for k in range(windows, 0, -1):
        end = len(history) - (k - 1) * h
        past, actual = history[:end - h], history[end - h:end]
        frame = pd.DataFrame({"window": k, "step": np.arange(1, h + 1), "actual": actual,
                              "benchmark": benchmark_forecasts(past, h, M)[BENCHMARK]})
        for name, member in members.items():
            fc = fit_member(member, past).predict(h=h, level=[80])
            frame[name], frame[f"{name} lo"], frame[f"{name} hi"] = fc["mean"], fc["lo-80"], fc["hi-80"]
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def combine(bt: pd.DataFrame, names: list[str], label: str) -> None:
    # Equal-weight average of the members' forecasts, and the average of their bounds.
    bt[label] = bt[names].mean(axis=1)
    bt[f"{label} lo"] = bt[[f"{n} lo" for n in names]].mean(axis=1)
    bt[f"{label} hi"] = bt[[f"{n} hi" for n in names]].mean(axis=1)


def window_table(bt: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # MASE of each forecast in each window, the mean over windows, and how often it was the best.
    table = bt.groupby("window").apply(
        lambda g: pd.Series({c: mase(g["actual"], g[c]) for c in columns}), include_groups=False).T
    table.columns = [f"window {w}" + (" (validation)" if w == 1 else "") for w in table.columns]
    table["mean"] = table.mean(axis=1)
    best = bt.groupby("window").apply(
        lambda g: min(columns, key=lambda c: mase(g["actual"], g[c])), include_groups=False)
    table["windows won"] = [int((best == c).sum()) for c in columns]
    return table


def winkler(actual, lo, hi, alpha=0.2) -> float:
    # Width of the interval plus 2/alpha times every miss (lecture, Week 8), in units of SCALE.
    actual, lo, hi = map(np.asarray, (actual, lo, hi))
    return float(np.mean((hi - lo) + 2 / alpha * (np.maximum(lo - actual, 0) + np.maximum(actual - hi, 0))) / SCALE)


def conformal_radius(errors) -> float:
    # The 80th percentile of the absolute errors: the forecast plus or minus this is the 80% interval.
    return float(np.quantile(np.abs(np.asarray(errors)), 0.80))


def interval_check(bt: pd.DataFrame, label: str) -> pd.DataFrame:
    # Both interval recipes, judged on windows they did not see: for each window the conformal radius
    # comes from the errors of the OTHER windows only.
    rows = []
    for w, g in bt.groupby("window"):
        others = bt[bt["window"] != w]
        r = conformal_radius(others["actual"] - others[label])
        for recipe, lo, hi in [("averaged bounds", g[f"{label} lo"], g[f"{label} hi"]),
                               ("conformal", g[label] - r, g[label] + r)]:
            rows.append({"recipe": recipe, "window": w,
                         "coverage": float(np.mean((g["actual"] >= lo) & (g["actual"] <= hi))),
                         "Winkler": winkler(g["actual"], lo, hi), "width": float(np.mean(hi - lo)) / SCALE})
    return pd.DataFrame(rows).groupby("recipe")[["coverage", "Winkler", "width"]].mean()

## 1. Your models, rebuilt — and one new member

*1 point.* Copy your **final** models from Lab 5 and Lab 7 — the ones you submitted — into `MEMBERS`, and
add MSTL. Then copy the accuracy scores you were sent after Labs 5 and 7 into `PAST`.

A member is `(kind, arguments)`:

- Lab 5: `("ets", {"spec": "MAM"})`, `("ets", {"spec": "AAdA"})` or `("ets", {"spec": None})` for `AutoETS`;
- Lab 7: `("arima", {"order": (0, 1, 1), "seasonal": (0, 1, 1), "log": True})` or
  `("arima", {"order": None, "log": False})` for `AutoARIMA`;
- MSTL: `("mstl", {"log": ...})` — the log decision is yours, the same as in Lab 7 or argued anew.

In [ ]:
MEMBERS = {
    "ETS (Lab 5)":    ("ets", {"spec": None}),                     # <-- your Lab 5 submission
    "ARIMA (Lab 7)":  ("arima", {"order": None, "log": False}),     # <-- your Lab 7 submission
    "MSTL":           ("mstl", {"log": False}),                     # <-- log or not
}

PAST = {                                     # <-- the scores you were sent: MASE and skill on the hidden horizon
    "Lab 5 (ETS)":            {"hidden MASE": None, "skill": None},
    "Lab 7 (seasonal ARIMA)": {"hidden MASE": None, "skill": None},
}

**Check that the rebuild is right.** The validation-window MASE below must match what your Lab 5 and Lab 7
notebooks printed for the same models (`valid MASE`). If a number differs, find out why before going on —
and say so in 3.3.

In [ ]:
rebuilt = pd.Series({name: mase(valid, fit_member(member, train).predict(h=H)["mean"])
                     for name, member in MEMBERS.items()}, name="validation MASE")
rebuilt["benchmark"] = mase(valid, simple[BENCHMARK])
rebuilt.to_frame()

**Do the rebuilt numbers match your earlier notebooks? What does MSTL score on the same window, next to
your two models?**

→

## 2. The combination

*3 points, scored from your submission file* on the hidden horizon against your benchmark:
skill = your MASE / your benchmark's MASE; below 1 — 3 points, up to 1.1 — 2, above — 1.

What you submit is an **equal-weight average of two or three members** from `MEMBERS`, with one of the two
intervals below. No single model, no estimated weights, no new model families.

### 2.1 Every member on four rolling origins

Run once — it refits every member four times (a few minutes for a monthly ARIMA).

In [ ]:
bt = backtest(MEMBERS, y)
names = list(MEMBERS)
combine(bt, names, "average of all")

### 2.2 Combine or select

**Before you look at the table**, write down which members go into your combination and why — each must be
defensible on its own (the lecture: a combination is as good as the worst member it cannot outvote). Then
read the table.

**My combination, decided before the table, and why.**

→

In [ ]:
COMBINATION = names                       # <-- the members you average: two or three of `names`
if COMBINATION != names:
    combine(bt, COMBINATION, "my combination")
LABEL = "my combination" if COMBINATION != names else "average of all"

columns = names + (["average of all"] if LABEL == "average of all" else ["average of all", LABEL]) + ["benchmark"]
table = window_table(bt, columns)
selection = table.loc[names, "mean"].idxmin()
print(f"selection on the rolling mean would have chosen: {selection}")
table

In [ ]:
# Do the members make different mistakes? A combination gains most when their errors are not correlated.
errors = pd.DataFrame({n: bt["actual"] - bt[n] for n in names})
print("correlation of the members' errors over all windows:")
errors.corr().round(2)

In [ ]:
# TODO: one plot of your choice that shows how the members and the combination behave across the windows
# (for example: each window's actual values with the members' forecasts, or MASE by window as lines).

### 2.3 The interval

Two recipes for the combination's 80% interval, each judged on windows it did not see:

- **averaged bounds** — the mean of the members' own lower and upper bounds;
- **conformal** — the combination's forecast ± the 80th percentile of its absolute errors on the *other*
  windows.

In [ ]:
interval_check(bt, LABEL)

### 2.4 The submission

Refit the members of your combination on the **whole** history `y`, average them, and build the interval
you chose. For the conformal interval the radius comes from **all four** windows of `bt`.

In [ ]:
# TODO: your final forecast, fitted on all of y.
#
#   final = {n: fit_member(MEMBERS[n], y).predict(h=H, level=[80]) for n in COMBINATION}
#   point = np.mean([final[n]["mean"] for n in COMBINATION], axis=0)
#   averaged bounds: np.mean([final[n]["lo-80"] ...]), np.mean([final[n]["hi-80"] ...])
#   conformal:       r = conformal_radius(bt["actual"] - bt[LABEL]); point - r, point + r
YHAT = None          # np.array of length H
YHAT_LO_80 = None    # np.array of length H
YHAT_HI_80 = None    # np.array of length H

for name, value in [("YHAT", YHAT), ("YHAT_LO_80", YHAT_LO_80), ("YHAT_HI_80", YHAT_HI_80)]:
    if value is None or len(np.ravel(value)) != H:
        raise ValueError(f"{name} must be an array of length H = {H}")

## 3. Justification block

*3 points, and the part a human reads closely.* Roughly a paragraph each. Numbers from your own cells,
not adjectives.

**3.1 Combine or select** *(1)* — the rolling table read: which member won how many windows, where the
combination landed in each window, and what the error correlations say about how much there was to gain.
What selection would have submitted, and why you combined anyway — or why your combination leaves a member
out.

→

**3.2 The interval** *(1)* — the two recipes' coverage, Winkler score and width on unseen windows; which
you submitted and why. How many errors your conformal radius rests on, and what that means for trusting it.

→

**3.3 Conclusions against your past results** *(1)* — your Lab 5 and Lab 7 scores on the hidden horizon
(`PAST`) next to what the rolling table says about the combination. Did combining help on your series, and
by how much against each of your earlier models? The skill you expect tonight, and why. One thing you would
now do differently in Lab 5 or Lab 7.

→

In [ ]:
# Optional, for 3.3: your past results and tonight's evidence in one table.
past = pd.DataFrame(PAST).T
past

## 4. Code quality and reproducibility

*1 point.* Before you export:

- Restart the kernel and run everything top to bottom. It must finish without errors.
- The submission file is written by the cell below, not edited by hand.
- No leakage: members and recipes are compared on the rolling windows; only the final members see all of
  `y`.
- Every number in your text comes from a cell.

## 5. Submit

Run both cells, then upload **three files** to the Week 9 activity on Moodle before 23:59:

- `SURNAME_lab09_forecast.csv` — written below;
- `SURNAME_lab09.pdf` — `File → Print → Save as PDF`;
- `SURNAME_lab09.ipynb` — `File → Download → Download .ipynb`.

In [ ]:
submission = pd.DataFrame({
    "student_id": MY_ID,
    "code": CODE,
    "ds": FUTURE.strftime("%Y-%m-%d"),
    "yhat": np.ravel(YHAT).astype(float),
    "yhat_lo_80": np.ravel(YHAT_LO_80).astype(float),
    "yhat_hi_80": np.ravel(YHAT_HI_80).astype(float),
})

name = f"{SURNAME.strip().upper()}_lab09_forecast.csv"
submission.to_csv(name, index=False)
print(submission.to_string(index=False))

In [ ]:
# The same checks used when your file is read. If this cell passes, your file is fine.
check = pd.read_csv(name, parse_dates=["ds"])
assert len(check) == H, f"expected {H} rows, got {len(check)}"
assert list(check.columns) == ["student_id", "code", "ds", "yhat", "yhat_lo_80", "yhat_hi_80"]
assert (check["ds"].to_numpy() == FUTURE.to_numpy()).all(), "the dates are not the H dates asked for"
assert check[["yhat", "yhat_lo_80", "yhat_hi_80"]].notna().all().all(), "no NaN allowed"
assert (check["yhat_lo_80"] <= check["yhat"]).all() and (check["yhat"] <= check["yhat_hi_80"]).all()
print("format OK -", name, "is ready to upload")

try:
    from google.colab import files
    files.download(name)
except ImportError:
    print(f"Not in Colab - {name} is saved next to this notebook.")